# OCR Date Extraction – Region-Based Approach

Goal: improve OCR accuracy for extracting event dates from flyer images.

This notebook records the region/crop/preprocessing experiments, then
**demonstrates the pipeline that now lives in `src/`**. It no longer defines
the pipeline itself:

| stage | lives in |
| --- | --- |
| crop + preprocess + OCR | `src/ocr.py` |
| OCR text cleanup | `src/ocr.py` (`clean_ocr_text`) |
| date extraction | `src/dates.py` (`extract_date`) |
| date normalization | `src/dates.py` (`normalize_date`) |
| all four, composed | `src/pipeline.py` (`extract_event_date`) |

## Setup

In [ ]:
import sys
from datetime import date
from pathlib import Path

from PIL import Image
import matplotlib.pyplot as plt
import pytesseract

# Make the project root importable so `src` resolves whether Jupyter was
# started from the repo root or from notebooks/.
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ocr import clean_ocr_text, crop_date_region, image_to_text, load_image, preprocess
from src.dates import extract_date, normalize_date
from src.pipeline import extract_event_date

image_dir = ROOT / "data" / "sample"
SAMPLE_IMAGE = image_dir / "cherry_blossom_market.jpeg"

# Flyers rarely print a year, so the normalizer infers one relative to a
# reference date. Fixed here so this notebook reproduces the documented
# result instead of changing with the calendar.
REFERENCE_TODAY = date(2026, 1, 1)

## Region Scan Experiments

In [ ]:
"""
Test likely flyer regions such as top, middle, and bottom to see where date text appears.
"""

for img_path in sorted(image_dir.glob("*")):
    print(f"\nProcessing: {img_path}")

    img = Image.open(img_path)

    width, height = img.size

    regions = {
        "top": (0, 0, width, int(height * 0.3)),
        "middle": (0, int(height * 0.3), width, int(height * 0.7)),
        "bottom": (0, int(height * 0.7), width, height),
    }

    # Show full image
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    # loop through regions
    for region_name, coords in regions.items():
        crop = img.crop(coords)

        print(f"--- {region_name} ---")

        plt.figure(figsize=(6, 3))
        plt.imshow(crop)
        plt.axis("off")
        plt.show()

## Preprocessing Experiments

In [ ]:
img = Image.open(SAMPLE_IMAGE)

width, height = img.size

regions = {
    "top": (0, 0, width, int(height * 0.3)),
    "middle": (0, int(height * 0.3), width, int(height * 0.7)),
    "bottom": (0, int(height * 0.7), width, height),
}

crop = img.crop(regions["middle"])

plt.figure(figsize=(8, 4))
plt.imshow(crop)
plt.axis("off")
plt.show()

crop_gray = crop.convert("L")
crop_big = crop_gray.resize((crop_gray.width * 2, crop_gray.height * 2))
crop_bw = crop_big.point(lambda p: 255 if p > 160 else 0)

text = pytesseract.image_to_string(crop_bw, config="--psm 6")

print("----- MIDDLE CROP OCR OUTPUT -----")
print(repr(text))

## Test Text Extraction on tighter middle cropped text

In [ ]:
middle_crop = img.crop(regions["middle"])
date_crop = middle_crop.crop((50, 140, 900, 450))

plt.figure(figsize=(8, 4))
plt.imshow(date_crop)
plt.axis("off")
plt.show()

crop_gray = date_crop.convert("L")
crop_big = crop_gray.resize((date_crop.width * 2, date_crop.height * 2))

plt.figure(figsize=(8, 4))
plt.imshow(crop_big, cmap="gray")
plt.axis("off")
plt.show()

crop_bw = crop_big.point(lambda p: 255 if p > 210 else 0)

plt.figure(figsize=(8, 4))
plt.imshow(crop_bw, cmap="gray")
plt.axis("off")
plt.show()

text_gray = pytesseract.image_to_string(crop_big, config="--psm 6")
text_bw = pytesseract.image_to_string(crop_bw, config="--psm 6")

print("----- GRAY OCR OUTPUT -----")
print(repr(text_gray))

print("----- BW OCR OUTPUT -----")
print(repr(text_bw))

## Final Pipeline

The four stages, called individually. Each is an ordinary function that can be
imported and tested on its own; only `image_to_text` needs Tesseract.

In [ ]:
image = load_image(SAMPLE_IMAGE)

region = crop_date_region(image)        # 1. crop to the likely date region
region = preprocess(region)             # 2. grayscale + 2x upscale
raw_text = image_to_text(region)        # 3. OCR
clean_text = clean_ocr_text(raw_text)   # 4. collapse whitespace
extracted = extract_date(clean_text)    # 5. find a month-name date
normalized = normalize_date(extracted, today=REFERENCE_TODAY)   # 6. resolve the year

print("RAW       :", repr(raw_text))
print("CLEAN     :", repr(clean_text))
print("EXTRACTED :", extracted)
print("NORMALIZED:", normalized)

The same thing in one call, which is what a batch runner would use:

In [ ]:
result = extract_event_date(SAMPLE_IMAGE, today=REFERENCE_TODAY)

for key, value in result.items():
    # Show the image path relative to the repo, so this cell's saved output
    # does not bake in a local absolute path.
    if key == "image":
        value = Path(value).relative_to(ROOT).as_posix()
    print(f"{key}: {value}")

image: data/sample/cherry_blossom_market.jpeg
raw_text: -_ FRIDAY, MARCH 27t
4PM - 8PM
clean_text: -_ FRIDAY, MARCH 27t 4PM - 8PM
date_found: March 27
normalized_date: 2026-03-27
valid: True


## Final Findings

**Sample image:** `cherry_blossom_market.jpeg`

**Best region:** A tighter crop around the middle of the flyer successfully isolated the event date and time.

**Best preprocessing:** Grayscale preserved the text more effectively than binary thresholding. Thresholding introduced substantial OCR noise.

**OCR result:** Tesseract extracted the primary event information as:

`FRIDAY, MARCH 27t`
`4PM - 8PM`

The ordinal suffix was partially misread, but the month and day remained recoverable through post-processing.

**Conclusion:** A tighter middle crop combined with grayscale preprocessing reliably extracted the event date from the sample flyer.

---

*Removed in the `src/` refactor:* the red-channel, multi-crop-box, preprocessing-variant
and PSM-comparison experiments. They operated on a flyer that was deleted from the
repository, so they could no longer run. Their conclusions are recorded in `CHANGELOG.md`
(grayscale beats thresholding; PSM choice made little difference; a digit whitelist helped
numeric dates).